# 🚗 SmartCar — Documentation des fonctions

**Projet** : SmartCar · BUT GEII · IUT de Toulouse  
**Plateforme** : STM32 Nucléo (C++ / STM32duino)  
**Interface** : RemoteXY via Bluetooth  

---

Ce notebook documente chaque fonction du code source SmartCar : rôle, paramètres, logique, et bugs identifiés.

## Sommaire

| # | Fonction | Statut |
|---|---|---|
| 1 | `pwm_direction(x)` | ✅ OK |
| 2 | `PWM_propulsion(...)` | ✅ OK |
| 3 | `gestion_bargraphe(nivB)` | ✅ OK |
| 4 | `bargraphe_ameliore(nivB, cap_av)` | ✅ OK |
| 5 | `gestion_phares(freinage, lum)` | ⚠️ Bug |
| 6 | `gestion_buzzer(pres, loin)` | ⚠️ 2 Bugs |
| 7 | `recep_temperature()` | ⚠️ Incomplet |
| 8 | `setup()` | ✅ OK |
| 9 | `loop()` | ⚠️ Bug |

---
## Brochage et Timers

### Broches

| Alias | Broche | Rôle |
|---|---|---|
| `PWM_AVANT_PIN` | D4 | Propulsion avant |
| `PWM_ARRIERE_PIN` | D5 | Propulsion arrière |
| `PWM_DROITE_PIN` | D3 | Direction droite |
| `PWM_GAUCHE_PIN` | D6 | Direction gauche |
| `PHARES_AV_PIN` | D11 | Phares avant |
| `PHARES_AR_PIN` | D2 | Phares arrière |
| `CMD_GENE_AUDIO_PIN` | D9 | Buzzer |
| `PRES_PIN` | D7 | Capteur proximité proche |
| `LOIN_PIN` | D8 | Capteur proximité loin |
| `SENS_AR_PIN` | PA1 | Sens marche arrière |
| `NIV_BATT_PIN` | A4 | Niveau batterie |
| `SDA_PIN` | D14 | I2C SDA |
| `SCL_PIN` | D15 | I2C SCL |

### Timers PWM

| Timer | Usage | Prescaler | Overflow | Fréquence |
|---|---|---|---|---|
| TIM2 | Direction | 40 | 100 | 20 000 Hz |
| TIM3 | Propulsion | 40 | 100 | 20 000 Hz |
| TIM4 | Phares | 40 | 100 | 20 000 Hz |

> **Formule** : `F = 80 000 000 / (Prescaler × Overflow)`

---
## 1. `pwm_direction(int x)` ✅

### Rôle
Contrôle la **direction** du véhicule (gauche / droite) en ajustant le rapport cyclique PWM des deux canaux du **TIM2**.

### Paramètre
| Paramètre | Type | Description |
|---|---|---|
| `x` | `int` | Valeur joystick axe X : **-100** (gauche) à **+100** (droite) |

### Logique

```
Si -10 < x < 10  →  zone morte  →  DROITE=0, GAUCHE=0
Si x < -10       →  virage gauche  →  DROITE=pwm, GAUCHE=0
Si x > +10       →  virage droite  →  DROITE=0,   GAUCHE=pwm
```

La valeur `pwm` est obtenue par `map(abs(x), 0, 100, 0, 100)` → proportionnel à l'inclinaison du joystick.

> ✅ **Aucun bug détecté.**

In [ ]:
# Simulation Python de pwm_direction()

def map_value(x, in_min, in_max, out_min, out_max):
    return int((x - in_min) * (out_max - out_min) / (in_max - in_min) + out_min)

def pwm_direction(x):
    if -10 < x < 10:
        return {"DROITE": 0, "GAUCHE": 0, "etat": "tout droit (zone morte)"}
    pwm = map_value(abs(x), 0, 100, 0, 100)
    if x < 0:
        return {"DROITE": pwm, "GAUCHE": 0, "etat": f"virage gauche (pwm={pwm})"}
    else:
        return {"DROITE": 0, "GAUCHE": pwm, "etat": f"virage droite (pwm={pwm})"}

# Tests
for val in [-100, -50, -5, 0, 5, 50, 100]:
    r = pwm_direction(val)
    print(f"x={val:5d}  →  DROITE={r['DROITE']:3d}  GAUCHE={r['GAUCHE']:3d}  | {r['etat']}")

---
## 2. `PWM_propulsion(pres, loin, sens_ar, vitesse, connexion)` ✅

### Rôle
Gère l'**avance ou le recul** du véhicule en tenant compte de la connexion Bluetooth, du sens de marche et des capteurs de proximité.

### Paramètres
| Paramètre | Type | Description |
|---|---|---|
| `pres` | `bool` | Obstacle très proche (<40 cm) |
| `loin` | `bool` | Obstacle détecté mais loin |
| `sens_ar` | `bool` | Marche arrière demandée |
| `vitesse` | `int` | Rapport cyclique PWM (0–100) |
| `connexion` | `bool` | Bluetooth connecté |

### Priorités (ordre décroissant)

```
1. !connexion  →  AVANT=0, ARRIERE=0  (stop total)
2. sens_ar     →  AVANT=0, ARRIERE=vitesse
3. pres        →  AVANT=0, ARRIERE=0  (obstacle proche)
4. loin        →  AVANT=vitesse/2, ARRIERE=0
5. sinon       →  AVANT=vitesse, ARRIERE=0
```

> ✅ **Aucun bug détecté.**

In [ ]:
# Simulation Python de PWM_propulsion()

def PWM_propulsion(pres, loin, sens_ar, vitesse, connexion):
    if not connexion:
        return {"AVANT": 0, "ARRIERE": 0, "etat": "stop — pas de connexion"}
    if sens_ar:
        return {"AVANT": 0, "ARRIERE": vitesse, "etat": "marche arrière"}
    if pres:
        return {"AVANT": 0, "ARRIERE": 0, "etat": "stop — obstacle proche"}
    elif loin:
        return {"AVANT": vitesse // 2, "ARRIERE": 0, "etat": "ralenti — obstacle loin"}
    else:
        return {"AVANT": vitesse, "ARRIERE": 0, "etat": "pleine vitesse"}

# Tests représentatifs
scenarios = [
    (False, False, False, 80, False),  # pas de connexion
    (False, False, True,  80, True),   # marche arrière
    (True,  False, False, 80, True),   # obstacle proche
    (False, True,  False, 80, True),   # obstacle loin
    (False, False, False, 80, True),   # voie libre
]

for pres, loin, ar, v, cnx in scenarios:
    r = PWM_propulsion(pres, loin, ar, v, cnx)
    print(f"pres={int(pres)} loin={int(loin)} ar={int(ar)} cnx={int(cnx)}  "
          f"→  AV={r['AVANT']:3d}  AR={r['ARRIERE']:3d}  | {r['etat']}")

---
## 3. `gestion_bargraphe(uint32_t nivB)` ✅

### Rôle
Convertit la valeur brute du **CAN 12 bits** (0–4095) en **pourcentage** (0–100) pour le bargraph de l'interface RemoteXY.

### Paramètre
| Paramètre | Type | Description |
|---|---|---|
| `nivB` | `uint32_t` | Valeur brute CAN de la tension batterie |

### Seuils et interpolation

| Valeur CAN | Tension équivalente | Bargraph |
|---|---|---|
| ≤ 3102 | ≤ 2.5V | 0% |
| ≥ 3723 | ≥ 3.0V | 100% |
| Entre les deux | — | `0.16117 × nivB − 500` |

> **Calcul des seuils** : `valeur_CAN = (4095 × tension) / 3.3`  
> 2.5V → `(4095 × 2.5) / 3.3 = 3102`  
> 3.0V → `(4095 × 3.0) / 3.3 = 3723`

> ✅ **Aucun bug détecté.**

In [ ]:
# Simulation Python de gestion_bargraphe() + visualisation

def gestion_bargraphe(nivB):
    x, y = 3102, 3723
    if nivB <= x:
        return 0
    elif nivB >= y:
        return 100
    else:
        return round(0.16117 * nivB - 500, 1)

# Affichage tabulaire
print(f"{'Valeur CAN':>12}  {'Tension (V)':>12}  {'Bargraph (%)':>13}")
print("-" * 42)
for can in range(2800, 4096, 100):
    tension = round(can * 3.3 / 4095, 3)
    barre = gestion_bargraphe(can)
    bar = "█" * int(barre // 5)
    print(f"{can:>12}  {tension:>12.3f}  {barre:>6.1f}%  {bar}")

---
## 4. `bargraphe_ameliore(uint32_t nivB, uint32_t cap_av)` ✅

### Rôle
Version améliorée du bargraph : affiche la **proximité de l'obstacle avant** en priorité, et revient au **niveau batterie** quand la voie est libre.

### Paramètres
| Paramètre | Type | Description |
|---|---|---|
| `nivB` | `uint32_t` | Valeur brute CAN de la batterie |
| `cap_av` | `uint32_t` | Valeur brute CAN du capteur de distance avant |

### Logique

| cap_av | Tension équivalente | Affichage |
|---|---|---|
| ≤ 930 | ≤ 0.75V (très proche) | 0% |
| > 3723 | > 3.0V (loin / libre) | Niveau batterie via `gestion_bargraphe()` |
| Entre les deux | — | `0.033 × cap_av − 30.69` |

> ✅ **Aucun bug détecté.**

In [ ]:
# Simulation Python de bargraphe_ameliore()

def bargraphe_ameliore(nivB, cap_av):
    if cap_av <= 930:
        return 0, "obstacle très proche"
    elif cap_av > 3723:
        return gestion_bargraphe(nivB), "voie libre → batterie"
    else:
        return round(0.033 * cap_av - 30.69, 1), "proximité intermédiaire"

# Test avec batterie à 80% (CAN~3600) et différentes distances
print(f"{'cap_av':>8}  {'Tension (V)':>12}  {'Bargraph':>10}  Contexte")
print("-" * 55)
for cap in [500, 930, 1500, 2500, 3500, 3723, 4000]:
    tension = round(cap * 3.3 / 4095, 3)
    barre, ctx = bargraphe_ameliore(3600, cap)
    print(f"{cap:>8}  {tension:>12.3f}  {barre:>9.1f}%  {ctx}")

---
## 5. `gestion_phares(bool freinage, bool lum)` ⚠️

### Rôle
Contrôle l'intensité des **phares avant** (freinage) et **arrière** (luminosité ambiante) via PWM.

### Paramètres
| Paramètre | Type | Description |
|---|---|---|
| `freinage` | `bool` | `true` → phares AV à 100% (255/255), `false` → 30% (77/255) |
| `lum` | `bool` | `true` → phares AR à 30%, `false` → éteints |

### Bug détecté 🔴

La variable `intensite_ar` est calculée mais **jamais envoyée au pin**.

```cpp
// ❌ Code original — ligne manquante
uint8_t intensite_ar = lum ? 30 : 0;
// analogWrite(PHARES_AR_PIN, intensite_ar);  ← ABSENT

// ✅ Code corrigé
uint8_t intensite_ar = lum ? 30 : 0;
analogWrite(PHARES_AR_PIN, intensite_ar);  // ← à ajouter
```

> ⚠️ **Les phares arrière ne s'allument jamais dans le code original.**

In [ ]:
# Simulation Python de gestion_phares()

def gestion_phares(freinage, lum):
    intensite_av = 255 if freinage else 77
    intensite_ar = 30 if lum else 0
    
    pct_av = round(intensite_av / 255 * 100)
    pct_ar = round(intensite_ar / 100 * 100)  # overflow=100 pour TIM4
    
    return {
        "phares_AV": intensite_av,
        "phares_AV_pct": f"{pct_av}%",
        "phares_AR": intensite_ar,
        "phares_AR_pct": f"{pct_ar}%",
        "bug": "analogWrite(PHARES_AR_PIN, ...) MANQUANT dans le code original"
    }

print("Scénario 1 — Freinage, nuit (lum=True)")
r = gestion_phares(True, True)
for k, v in r.items(): print(f"  {k}: {v}")

print("\nScénario 2 — Roulage normal, jour")
r = gestion_phares(False, False)
for k, v in r.items(): print(f"  {k}: {v}")

---
## 6. `gestion_buzzer(bool pres, bool loin)` ⚠️

### Rôle
Gère le buzzer de proximité avec une logique **non-bloquante** basée sur `millis()`.

### Paramètres
| Paramètre | Type | Description |
|---|---|---|
| `pres` | `bool` | Obstacle très proche → 4 bips/sec (interval=125ms) |
| `loin` | `bool` | Obstacle loin → 2 bips/sec (interval=250ms) |

### Comportement attendu
| Condition | Comportement |
|---|---|
| `klaxon == 1` | Buzzer continu (prioritaire) |
| `pres == true` | 4 bips/sec |
| `loin == true` | 2 bips/sec |
| Aucun obstacle | Silence |

### Bugs détectés 🟠

**Bug 1 — `loin` et `else` inversés :**
```cpp
// ❌ Code original
else if (loin)  { digitalWrite(LOW); }   // devrait être interval=250
else            { interval = 250; }       // devrait être digitalWrite(LOW)

// ✅ Code corrigé
else if (loin)  { interval = 250; }      // 2 bips/sec
else            { digitalWrite(CMD_GENE_AUDIO_PIN, LOW); return; }  // silence
```

**Bug 2 — `buzzerState` non réinitialisé :**
```cpp
// ❌ Code original
if (klaxon == 1) { buzzerState = true; } // reste true après relâchement

// ✅ Code corrigé : reset dans le else
```

> ⚠️ **2 bugs logiques — le buzzer ne se comporte pas correctement.**

In [ ]:
# Simulation Python de gestion_buzzer() — version corrigée
import time

class BuzzerSimulator:
    def __init__(self):
        self.temps_precedent = time.time() * 1000
        self.buzzer_state = False

    def tick(self, pres, loin, klaxon):
        temps_actuel = time.time() * 1000
        temps_ecoule = temps_actuel - self.temps_precedent
        interval = 0

        if klaxon:
            self.buzzer_state = True
        else:
            if pres:
                interval = 125    # 4 bips/sec
            elif loin:
                interval = 250    # 2 bips/sec
            else:
                self.buzzer_state = False  # silence
                return self.buzzer_state
            if temps_ecoule >= interval:
                self.temps_precedent = temps_actuel
                self.buzzer_state = not self.buzzer_state
        return self.buzzer_state

bz = BuzzerSimulator()
print("Simulation 10 ticks — obstacle proche (pres=True)")
for i in range(10):
    time.sleep(0.13)
    etat = bz.tick(pres=True, loin=False, klaxon=False)
    print(f"  tick {i+1:2d} → buzzer {'ON 🔊' if etat else 'OFF  '}")

---
## 7. `recep_temperature(void)` ⚠️

### Rôle
Lit la température ambiante depuis un **capteur I2C compatible LM75** (adresse `0x48`).

### Protocole I2C

```
STM32                          LM75 (0x48)
  │─── beginTransmission(0x48) ──►│
  │─── write(0x00) ───────────────►│  sélectionne le registre température
  │─── endTransmission(false) ────►│  repeated start
  │─── requestFrom(0x48, 2) ──────►│
  │◄── MSB (partie entière) ───────│  ex: 0x18 = 24°C
  │◄── LSB (fraction) ─────────────│  bit 7 = 0.5°C
```

### Format des données reçues

| Octet | Bits | Signification |
|---|---|---|
| MSB | [7..0] | Température entière signée (complément à 2) |
| LSB | [7] | Fraction (1 = +0.5°C) |

### État du code ⚠️

```cpp
// ❌ LSB commenté → pas de lecture fractionnaire
// temp_Ambiante_frac = Wire.read();

// ❌ Envoi RemoteXY commenté → température jamais affichée
// RemoteXY.temperature = temp_Ambiante_entier;
```

> ⚠️ **Fonction incomplète : la température n'est pas transmise à l'interface.**

In [ ]:
# Décodage des données LM75 (format I2C)

def decode_lm75(msb_raw, lsb_raw):
    """
    Décode les 2 octets reçus du capteur LM75.
    msb_raw : octet MSB (partie entière, complément à 2 sur 8 bits)
    lsb_raw : octet LSB (bit 7 = fraction 0.5°C)
    """
    # Partie entière (signée)
    if msb_raw > 127:
        entier = msb_raw - 256  # négatif
    else:
        entier = msb_raw
    
    # Fraction
    fraction = 0.5 if (lsb_raw & 0x80) else 0.0
    
    return entier + fraction

# Exemples
exemples = [
    (0x18, 0x00, "24.0°C"),
    (0x18, 0x80, "24.5°C"),
    (0x00, 0x80, "0.5°C"),
    (0xFF, 0x00, "-1.0°C"),
    (0xEC, 0x00, "-20.0°C"),
]

print(f"{'MSB':>6}  {'LSB':>6}  {'Décodé':>10}  {'Attendu':>10}")
print("-" * 40)
for msb, lsb, attendu in exemples:
    t = decode_lm75(msb, lsb)
    ok = "✅" if str(t) in attendu else "❌"
    print(f"  0x{msb:02X}   0x{lsb:02X}  {t:>9.1f}°C  {attendu:>10}  {ok}")

---
## 8. `setup()` ✅

### Rôle
Initialisation **unique au démarrage**. Configure tout le matériel avant l'entrée dans la boucle principale.

### Étapes d'initialisation

| Étape | Appel | Description |
|---|---|---|
| 1 | `RemoteXY_Init()` | Démarrage communication Bluetooth |
| 2 | `pinMode(OUTPUT/INPUT)` | Configuration GPIO |
| 3 | `analogReadResolution(12)` | CAN 12 bits (0–4095) |
| 4 | `timerDirection` | TIM2 — Prescaler=40, Overflow=100 |
| 5 | `timerPropulsion` | TIM3 — Prescaler=40, Overflow=100 |
| 6 | `timerPhares` | TIM4 — Prescaler=40, Overflow=100 |
| 7 | `Wire.begin()` | I2C sur D14 (SDA) / D15 (SCL) |

> ✅ **Aucun bug détecté.**

In [ ]:
# Vérification des fréquences PWM configurées dans setup()

F_CLK = 80_000_000  # 80 MHz (STM32 Nucléo)

timers = [
    ("TIM2", "Direction",  40, 100),
    ("TIM3", "Propulsion", 40, 100),
    ("TIM4", "Phares",     40, 100),
]

print(f"{'Timer':>6}  {'Usage':>12}  {'Prescaler':>10}  {'Overflow':>9}  {'Fréquence':>12}")
print("-" * 58)
for nom, usage, prescaler, overflow in timers:
    freq = F_CLK / (prescaler * overflow)
    print(f"{nom:>6}  {usage:>12}  {prescaler:>10}  {overflow:>9}  {freq:>10.0f} Hz")

---
## 9. `loop()` ⚠️

### Rôle
**Boucle principale** exécutée en permanence. Lit les capteurs, met à jour l'interface RemoteXY, pilote le véhicule.

### Ordre d'exécution

```
1. RemoteXYEngine.handler()     → échange Bluetooth
2. digitalRead(PRES_PIN)        → lecture capteur proche
3. digitalRead(LOIN_PIN)        → lecture capteur loin
4. analogRead(NIV_BATT_PIN)     → lecture batterie
5. [jeu de test hardcodé]       → pres=1, loin=0, klaxon=0
6. gestion_bargraphe(nivBat)    → mise à jour bargraph
7. gestion_buzzer(pres, loin)   → mise à jour buzzer
8. pwm_direction(joystick_x)    → pilotage direction
9. gestion_phares(freinage, lum)→ pilotage phares
```

### Bug détecté 🟠

```cpp
// ❌ freinage et lum utilisés sans être initialisés
bool freinage;  // valeur indéterminée
bool lum;       // valeur indéterminée
gestion_phares(freinage, lum);  // comportement imprévisible

// ✅ Correction
bool freinage = false;
bool lum = false;
```

> ⚠️ **`freinage` et `lum` non initialisés → comportement indéterminé.**

---
## 🐛 Récapitulatif des bugs

| # | Fonction | Sévérité | Description | Correction |
|---|---|---|---|---|
| 1 | `gestion_phares` | 🔴 Majeur | `analogWrite(PHARES_AR_PIN, ...)` absent | Ajouter la ligne manquante |
| 2 | `gestion_buzzer` | 🟠 Moyen | `loin` et `else` inversés | Échanger les deux blocs |
| 3 | `gestion_buzzer` | 🟠 Moyen | `buzzerState` non réinitialisé après klaxon | Ajouter reset dans le `else` |
| 4 | `recep_temperature` | 🟡 Mineur | LSB et affichage RemoteXY commentés | Décommenter et compléter |
| 5 | `loop` | 🟠 Moyen | `freinage` et `lum` non initialisés | Initialiser à `false` |